In [24]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random

%matplotlib inline


In [25]:
words = open("names.txt",mode='r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [26]:
block_size = 3

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f"X.shape{X.shape}")
    print(f"Y.shape{Y.shape}")

    return X,Y

random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xval,Yval = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])



X.shapetorch.Size([182625, 3])
Y.shapetorch.Size([182625])
X.shapetorch.Size([22655, 3])
Y.shapetorch.Size([22655])
X.shapetorch.Size([22866, 3])
Y.shapetorch.Size([22866])


In [27]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((vocab_size,n_embd))    #查找表
W1 = torch.randn((n_embd*block_size,n_hidden),generator=g)*(5/3)/((n_embd*block_size))**0.5
#b1 = torch.randn(n_embd*block_size,n_hidden,generator=g)
W2 = torch.randn((n_hidden,vocab_size),generator=g) * 0.01
b2 = torch.randn(vocab_size,generator=g) * 0

#BatchNorm Parameters
bngain = torch.ones((1,n_hidden))
bnbias = torch.zeros((1,n_hidden))

bnmean_running = torch.ones((1,n_hidden))
bnstd_running = torch.zeros((1,n_hidden))

parameters = [C,W1,W2,b2,bngain,bnbias]

print(f"The total num of parameters:{sum(p.nelement() for p in parameters)}")

for p in parameters:
    p.requires_grad = True

The total num of parameters:12097


In [28]:
max_step = 20000
batch_size = 32

lossi = []

print_interval = 500  # 每500步打印一次

for i in range(max_step):
    #Mini-Batch
    ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator=g)
    Xb,Yb = Xtr[ix],Ytr[ix]

    emb = C[Xb]
    embcat = emb.view(emb.shape[0],-1)

    hpreact = embcat @ W1

    #------------------------------------------
    #BatchNorm Layer

    bnmeani = hpreact.mean(0,keepdim=True)
    bnstdi = hpreact.std(0,keepdim=True)

    #Normalization and ScalingShifting
    hpreact = (hpreact-bnmeani) / bnstdi
    hpreact = hpreact*bngain + bnbias

    with torch.no_grad():
        bnmean_running = 0.999*bnmean_running + 0.001*bnmeani
        bnstd_running = 0.999*bnstd_running + 0.001*bnstdi

    #Non-Linear
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits,Yb)

    lossi.append(loss.item())

    loss.backward()

    #update
    lr = 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad
        p.grad = None  # 清空梯度，防止累积

    # 调试输出
    if i % print_interval == 0 or i == max_step - 1:
        print(f"Step {i:5d} | Loss: {loss.item():.4f}")


Step     0 | Loss: 3.3042
Step   500 | Loss: 2.5168
Step  1000 | Loss: 2.6670
Step  1500 | Loss: 2.2617
Step  2000 | Loss: 2.0136
Step  2500 | Loss: 2.4914
Step  3000 | Loss: 2.7501
Step  3500 | Loss: 2.2153
Step  4000 | Loss: 2.1058
Step  4500 | Loss: 2.7224
Step  5000 | Loss: 2.3770
Step  5500 | Loss: 2.3235
Step  6000 | Loss: 2.5386
Step  6500 | Loss: 2.1096
Step  7000 | Loss: 2.3507
Step  7500 | Loss: 2.5076
Step  8000 | Loss: 2.2939
Step  8500 | Loss: 2.7688
Step  9000 | Loss: 1.9419
Step  9500 | Loss: 2.2588
Step 10000 | Loss: 2.4273
Step 10500 | Loss: 2.3621
Step 11000 | Loss: 2.1072
Step 11500 | Loss: 2.2145
Step 12000 | Loss: 2.4575
Step 12500 | Loss: 2.1457
Step 13000 | Loss: 2.4711
Step 13500 | Loss: 2.3911
Step 14000 | Loss: 1.9398
Step 14500 | Loss: 2.2064
Step 15000 | Loss: 2.4384
Step 15500 | Loss: 2.2706
Step 16000 | Loss: 2.5317
Step 16500 | Loss: 2.4456
Step 17000 | Loss: 2.3063
Step 17500 | Loss: 1.9262
Step 18000 | Loss: 2.0409
Step 18500 | Loss: 2.0358
Step 19000 |

In [30]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        'train':(Xtr,Ytr),
        'val':(Xval,Yval),
        'Xte':(Xte,Yte)
    }[split]

    emb = C[x]
    embcat = emb.view(emb.shape[0],-1)
    hpreact = embcat @ W1

    #推理阶段，不必再计算样本均值和方差了，直接使用之前留存的running参数
    hpreact = bngain * (hpreact-bnmean_running) / bnstd_running + bnbias

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits,y)
    print(split,loss.item())

split_loss('train')
split_loss('val')


train 2.2064948081970215
val 2.22121000289917


In [49]:
#构建封装类，实现核心组件

#--------------线性层----------------
class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight = torch.randn((fan_in,fan_out),generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    

class BatchNorm1d:
    def __init__(self,dim,eps = 1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        #可学习的参数
        self.gamma = torch.ones(dim)    #bngain
        self.beta = torch.zeros(dim)    #bnbias

        #全局平滑统计量
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):
        #需要做分流逻辑处理，判断当前是训练还是推理，采用不同的计算方式
        if self.training:
            xmean = x.mean(0,keepdim=True)
            xvar = x.var(0,keepdim= True)
        else:
            xmean = x.running_mean
            xvar = x.running_var

        xhat = (x-xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out
    
    def parameters(self):
        return [self.gamma,self.beta]
    
class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []
    


In [50]:
#组装深层网络
n_embd = 10 
n_hidden = 100 

# 构建词向量矩阵
C = torch.randn((vocab_size, n_embd), generator=g)

# 像搭积木一样，把组件按顺序放进一个列表里
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size)
]

# 统一提取所有参数
parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
  p.requires_grad = True



x = embcat
for layer in layers:
  x = layer(x)
loss = F.cross_entropy(x, Yb) # x 现在就是最后的 logits
